# Generate Supplementary Tables (S3-S6) and Data-Specific Reports

This notebook cell:
- reads each dataset from `raw_files/`
- writes output TSV files into `files/` as `Supplementary_Table3.tsv`, `Supplementary_Table4.tsv`, ...
- creates a data-spec report in Markdown with variable and missing-data details for each generated table.

In [3]:
from pathlib import Path
import pandas as pd
import yaml


def find_base_dir() -> Path:
    # Try current dir and parents first.
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "raw_files").exists():
            return candidate

    # Fallback to this repository's expected location.
    fallback = cwd / "analysis" / "supplementary_data"
    if (fallback / "raw_files").exists():
        return fallback

    raise FileNotFoundError(
        "Could not locate 'raw_files' directory. Run this notebook from the project root or analysis/supplementary_data."
    )


base_dir = find_base_dir()
raw_dir = base_dir / "raw_files"
out_dir = base_dir / "files"
out_dir.mkdir(parents=True, exist_ok=True)


def read_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file format: {path.name}")


def load_variable_dictionary(dict_path: Path) -> dict:
    if not dict_path or not dict_path.exists():
        return {}
    with dict_path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def resolve_variable_dictionary(table_stem: str) -> tuple[Path | None, str]:
    final_path = out_dir / f"{table_stem}_variable_dictionary.yaml"
    review_path = out_dir / f"{table_stem}_variable_dictionary.review_copy.yaml"
    draft_path = out_dir / f"{table_stem}_variable_dictionary.draft.yaml"

    # During drafting, prefer the editable review copy over older final files.
    if review_path.exists():
        return review_path, "review_copy"
    if draft_path.exists():
        return draft_path, "draft"
    if final_path.exists():
        return final_path, "final"
    return None, "missing"


def format_variable_list(df: pd.DataFrame, metadata: dict) -> list[str]:
    lines = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        meta = metadata.get(col, {})

        description = meta.get("description", "not provided")
        unit = meta.get("unit", "not provided")
        allowed_range = meta.get("allowed_range", "not provided")
        review_status = meta.get("review_status", "missing")

        suffix = ""
        if review_status != "auto_inferred":
            suffix = f"; review_status={review_status}"

        lines.append(
            f"  - {col}: description={description}; unit={unit}; value labels/range={allowed_range}; dtype={dtype}{suffix}"
        )
    return lines


def summarize_missing_codes(df: pd.DataFrame) -> list[str]:
    # Common missing-value tokens frequently used in tabular datasets.
    candidate_codes = ["", "NA", "N/A", "NULL", "null", "None", "none", "Unknown", "unknown", "-999", "-99", "999"]
    lines = []

    for code in candidate_codes:
        count = (df.astype(str) == code).sum().sum()
        if count > 0:
            lines.append(f"  - '{code}': found {int(count)} value(s)")

    null_count = int(df.isna().sum().sum())
    if null_count > 0:
        lines.append(f"  - NaN/blank parsed by pandas: found {null_count} value(s)")

    if not lines:
        lines.append("  - none detected")

    return lines


def summarize_special_formats(df: pd.DataFrame) -> str:
    datetime_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
    categorical_cols = [c for c in df.columns if isinstance(df[c].dtype, pd.CategoricalDtype)]

    parts = []
    if datetime_cols:
        parts.append(f"datetime columns: {', '.join(datetime_cols)}")
    if categorical_cols:
        parts.append(f"categorical columns: {', '.join(categorical_cols)}")

    return "; ".join(parts) if parts else "none specified"


raw_files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
if not raw_files:
    raise FileNotFoundError(f"No input files found in: {raw_dir}")

if len(raw_files) != 4:
    print(f"Warning: expected 4 files for S3-S6, but found {len(raw_files)}. Numbering will continue from S3.")

report_lines = []
used_draft_sources = False

for idx, input_path in enumerate(raw_files, start=3):
    df = read_dataset(input_path)

    out_name = f"Supplementary_Table{idx}.tsv"
    out_path = out_dir / out_name
    df.to_csv(out_path, sep="\t", index=False)

    metadata_path, metadata_kind = resolve_variable_dictionary(input_path.stem)
    metadata = load_variable_dictionary(metadata_path)
    if metadata_kind != "final":
        used_draft_sources = True

    report_lines.append(f"# DATA-SPECIFIC INFORMATION FOR: [{out_name}]")
    report_lines.append("*repeat this section for each dataset, folder or file, as appropriate*")
    report_lines.append("")
    report_lines.append(f"* Number of variables: {df.shape[1]}")
    report_lines.append(f"* Number of cases/rows: {df.shape[0]}")
    report_lines.append("* Variable List: *list variable name(s), description(s), unit(s) and value labels as appropriate for each*")
    report_lines.extend(format_variable_list(df, metadata))
    if metadata_path:
        report_lines.append(f"* Variable dictionary source: {metadata_path.name} ({metadata_kind})")
    else:
        report_lines.append("* Variable dictionary source: not found (generated fallback descriptions used)")
    report_lines.append("* Missing data codes: *list code/symbol and definition*")
    report_lines.extend(summarize_missing_codes(df))
    report_lines.append(f"* Specialized formats or other abbreviations used: {summarize_special_formats(df)}")
    report_lines.append("")

report_name = "Supplementary_Data_Spec_Report.draft.md" if used_draft_sources else "Supplementary_Data_Spec_Report.md"
report_path = out_dir / report_name
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Generated {len(raw_files)} supplementary table(s) in: {out_dir}")
print(f"Report written to: {report_path}")

Generated 4 supplementary table(s) in: /home/tobamo/analize/project-tobamo/analysis/supplementary_data/files
Report written to: /home/tobamo/analize/project-tobamo/analysis/supplementary_data/files/Supplementary_Data_Spec_Report.draft.md
